In [ ]:
"""
    Sample notebook
"""

In [ ]:
# pylint: disable=unnecessary-pass
%pip install polars
%pip install altair
%pip install vegafusion

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: c:\Users\z004c84b\.pyenv\pyenv-win\versions\3.14.3\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: c:\Users\z004c84b\.pyenv\pyenv-win\versions\3.14.3\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: c:\Users\z004c84b\.pyenv\pyenv-win\versions\3.14.3\python.exe -m pip install --upgrade pip


In [2]:
import os
import polars as pl
import altair as alt
alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [15]:
os.chdir("C:/Users/z004c84b/Downloads")
WORK_DIR = os.getcwd()
print (WORK_DIR)

DATA = os.path.join(WORK_DIR, "")
print(DATA)

C:\Users\z004c84b\Downloads
C:\Users\z004c84b\Downloads\


In [32]:
# pl.scan_csv doesn't load the file into memory immediately but only when called with the
# .collect() method, which generally makes running the code significantly faster.
# schema_overrides is optional and can be used to explicitly set a data type to a column,
# but it will return an error if polars finds some kind of mismatch.
def get_data(file: str, separator: str = ",", schema_overrides: dict = None) -> pl.LazyFrame:
    """
    Scan csv file from path
    """
    return pl.scan_csv(
        source=f"{DATA}/{file}",
        separator=separator,
        schema_overrides=schema_overrides,
        decimal_comma=True
    )

In [8]:
yield_curve_scheme = {
    "date": pl.Date,
    "yield_percentage": pl.Float64,
    "maturity_month": pl.Float64
}

equity_data_scheme = {
    "Date": pl.Date
}

# pl.Datetime("ns") is accurate to the nano-second.
quotes_inc_eu_schema = {
    "side": pl.Utf8,
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns"),
    "price": pl.Float64
}

trades_eu_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns")
}

quotes_inc_us_schema = {
    "side": pl.Utf8,
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns"),
    "price": pl.Float64
}

trades_us_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns")
}

In [11]:
YIELD_CURVE_FILE = "yield_curve.csv"
EQUITY_DATA_FILE = "equity_data.csv"
QUOTES_INC_EU_FILE = "DE0007500001_quotes_incremental.csv"
TRADES_EU_FILE = "DE0007500001_trades.csv"
QUOTES_INC_US_FILE = "US2561631068_quotes_incremental.csv"
TRADES_US_FILE = "US2561631068_trades.csv"

In [19]:
# Yield Curve
# The original file was an .xlsx file, which I've modified to a .csv
# Also, I've slightly modified the files to a proper tabular format,
# adding the reference date (1989-03-31) as a column. The other values
# remain unchanged.
yield_curve = get_data(file = YIELD_CURVE_FILE,
                       separator = ";",
                       schema_overrides=yield_curve_scheme)
yield_curve = yield_curve.drop_nulls()

print(yield_curve.collect())
print(yield_curve.collect_schema())
print(yield_curve.collect().describe())

shape: (17, 3)
┌────────────┬──────────────────┬────────────────┐
│ date       ┆ yield_percentage ┆ maturity_month │
│ ---        ┆ ---              ┆ ---            │
│ date       ┆ f64              ┆ f64            │
╞════════════╪══════════════════╪════════════════╡
│ 1989-03-31 ┆ 9.12             ┆ 3.0            │
│ 1989-03-31 ┆ 9.32             ┆ 6.0            │
│ 1989-03-31 ┆ 9.34             ┆ 9.0            │
│ 1989-03-31 ┆ 9.62             ┆ 12.0           │
│ 1989-03-31 ┆ 9.69             ┆ 15.0           │
│ …          ┆ …                ┆ …              │
│ 1989-03-31 ┆ 9.25             ┆ 72.0           │
│ 1989-03-31 ┆ 9.15             ┆ 84.0           │
│ 1989-03-31 ┆ 9.12             ┆ 96.0           │
│ 1989-03-31 ┆ 9.05             ┆ 108.0          │
│ 1989-03-31 ┆ 9.0              ┆ 120.0          │
└────────────┴──────────────────┴────────────────┘
Schema({'date': Date, 'yield_percentage': Float64, 'maturity_month': Float64})
shape: (9, 4)
┌────────────┬───────────

In [20]:
alt.Chart(yield_curve.collect()).mark_line().encode(
    x=alt.X('maturity_month',
            title='Maturity in months'),
    y=alt.Y('yield_percentage',
            title='Yield in %',
            scale=alt.Scale(zero=False, padding=10))
)

alt.Chart(...)

In [21]:
# The original file was an .xlsx file which I've converted to csv.
equity_data = get_data(
    file = EQUITY_DATA_FILE,
    separator = ";",
    schema_overrides=equity_data_scheme
)

print(equity_data.collect())
print(equity_data.collect_schema())
print(equity_data.collect().describe())

shape: (1_304, 41)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Date      ┆ ADIDAS -  ┆ AIRBUS -  ┆ ALLIANZ - ┆ … ┆ SYMRISE - ┆ VOLKSWAGE ┆ VONOVIA - ┆ ZALANDO  │
│ ---       ┆ TOT       ┆ TOT       ┆ TOT       ┆   ┆ TOT       ┆ N PREF. - ┆ TOT       ┆ - TOT    │
│ date      ┆ RETURN    ┆ RETURN    ┆ RETURN    ┆   ┆ RETURN    ┆ TOT       ┆ RETURN    ┆ RETURN   │
│           ┆ IND       ┆ IND       ┆ IND       ┆   ┆ IND       ┆ RETURN …  ┆ IND       ┆ IND      │
│           ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2021-01-0 ┆ 4124.59   ┆ 760.55    ┆ 8424.11   ┆ … ┆ 835.99    ┆ 1655.24   ┆ 456.22    ┆ 423.63   │
│ 1         ┆           ┆           ┆           ┆   ┆           ┆       

In [ ]:
# To render multiple lines in a lineplot, y-columns are treated essentially as layers.
def plot_lines(data: pl.LazyFrame, x_col, y_cols, x_label=None):
    """
        Create a line chart with Altair.
    """
    chart = alt.Chart(data).mark_line().encode(
        x=alt.X(x_col,
                title=x_label or x_col),
        y=alt.Y(alt.repeat('layer'))
            .aggregate('mean')
            .title('Return(s)'),
        color=alt.ColorDatum(alt.repeat('layer')),
    ).repeat(
        layer=y_cols
    )

    return chart

In [23]:
equity_y_cols = ['AIRBUS - TOT RETURN IND', 'ADIDAS - TOT RETURN IND', 'ALLIANZ - TOT RETURN IND']
plot_lines(
    data=equity_data.collect(),
    x_col="Date",
    y_cols=equity_y_cols
)

alt.RepeatChart(...)

In [25]:
## EU: Quotes Incremental
quotes_inc_eu = get_data(
    file=QUOTES_INC_EU_FILE,
    schema_overrides=quotes_inc_eu_schema
)

print(quotes_inc_eu.collect().head())
print(quotes_inc_eu.collect_schema())
print(quotes_inc_eu.collect().describe())

shape: (5, 34)
┌──────┬───────┬───────┬───────────────┬───┬───────────────┬───────────────┬───────────────┬───────┐
│ side ┆ price ┆ size  ┆ order_id      ┆ … ┆ total_ask_ord ┆ total_bid_ord ┆ market_state  ┆ venue │
│ ---  ┆ ---   ┆ ---   ┆ ---           ┆   ┆ ers           ┆ ers           ┆ ---           ┆ ---   │
│ str  ┆ f64   ┆ i64   ┆ i64           ┆   ┆ ---           ┆ ---           ┆ str           ┆ str   │
│      ┆       ┆       ┆               ┆   ┆ i64           ┆ i64           ┆               ┆       │
╞══════╪═══════╪═══════╪═══════════════╪═══╪═══════════════╪═══════════════╪═══════════════╪═══════╡
│ BID  ┆ 7.074 ┆ 1828  ┆ 1081350376584 ┆ … ┆ 0             ┆ 1             ┆ CONTINUOUS_TR ┆ CEUX  │
│      ┆       ┆       ┆ 637819        ┆   ┆               ┆               ┆ ADING         ┆       │
│ ASK  ┆ 7.164 ┆ 1828  ┆ 1081350376584 ┆ … ┆ 1             ┆ 1             ┆ CONTINUOUS_TR ┆ CEUX  │
│      ┆       ┆       ┆ 637820        ┆   ┆               ┆               ┆

In [ ]:
## EU: Trades
trades_eu = get_data(
    file=TRADES_EU_FILE,
    schema_overrides=trades_eu_schema
)

print(trades_eu.collect().head())
print(trades_eu.collect_schema())
print(trades_eu.collect().describe())

shape: (5, 9)
┌────────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────┐
│ trade_id   ┆ trade_time ┆ publicatio ┆ aggressor ┆ … ┆ execution ┆ market_st ┆ trade_typ ┆ venue │
│ ---        ┆ stamp      ┆ n_timestam ┆ _side     ┆   ┆ _size     ┆ ate       ┆ e         ┆ ---   │
│ i128       ┆ ---        ┆ p          ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ str   │
│            ┆ str        ┆ ---        ┆ str       ┆   ┆ i64       ┆ str       ┆ str       ┆       │
│            ┆            ┆ str        ┆           ┆   ┆           ┆           ┆           ┆       │
╞════════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════╡
│ 6269133135 ┆ 2023-09-01 ┆ 2023-09-01 ┆ ASK       ┆ … ┆ 166       ┆ CONTINUOU ┆ LIT       ┆ CEUX  │
│ 937        ┆ 07:00:23.8 ┆ 07:00:23.8 ┆           ┆   ┆           ┆ S_TRADING ┆           ┆       │
│            ┆ 34308000   ┆ 34308000   ┆           ┆   ┆           ┆         

In [27]:
## US: Quotes Incremental
quotes_inc_us = get_data(
    file=QUOTES_INC_US_FILE,
    schema_overrides=quotes_inc_us_schema
)

print(quotes_inc_us.collect().head())
print(quotes_inc_us.collect_schema())
print(quotes_inc_us.collect().describe())

shape: (5, 37)
┌─────┬──────┬───────┬──────┬───┬──────────────────┬──────────────────┬──────────────┬───────┐
│     ┆ side ┆ price ┆ size ┆ … ┆ total_ask_orders ┆ total_bid_orders ┆ market_state ┆ venue │
│ --- ┆ ---  ┆ ---   ┆ ---  ┆   ┆ ---              ┆ ---              ┆ ---          ┆ ---   │
│ i64 ┆ str  ┆ f64   ┆ i64  ┆   ┆ i64              ┆ i64              ┆ str          ┆ str   │
╞═════╪══════╪═══════╪══════╪═══╪══════════════════╪══════════════════╪══════════════╪═══════╡
│ 0   ┆ BID  ┆ 35.41 ┆ 1878 ┆ … ┆ 0                ┆ 1                ┆ PRE_OPEN     ┆ XASE  │
│ 1   ┆ BID  ┆ 47.25 ┆ 100  ┆ … ┆ 0                ┆ 2                ┆ PRE_OPEN     ┆ XASE  │
│ 2   ┆ ASK  ┆ 54.18 ┆ 100  ┆ … ┆ 1                ┆ 2                ┆ PRE_OPEN     ┆ XASE  │
│ 3   ┆ BID  ┆ 50.16 ┆ 1500 ┆ … ┆ 1                ┆ 3                ┆ PRE_OPEN     ┆ XASE  │
│ 4   ┆ ASK  ┆ 50.9  ┆ 1500 ┆ … ┆ 2                ┆ 3                ┆ PRE_OPEN     ┆ XASE  │
└─────┴──────┴───────┴──────┴───┴──

In [28]:
## US: Trades
trades_us = get_data(
    file=TRADES_US_FILE,
    schema_overrides=trades_us_schema
)

print(trades_us.collect().head())
print(trades_us.collect_schema())
print(trades_us.collect().describe())

shape: (5, 15)
┌─────┬──────────┬─────────────┬─────────────┬───┬─────────────┬─────────────┬─────────────┬───────┐
│     ┆ trade_id ┆ trade_times ┆ publication ┆ … ┆ bmll_trade_ ┆ trade_actio ┆ execution_v ┆ venue │
│ --- ┆ ---      ┆ tamp        ┆ _timestamp  ┆   ┆ type        ┆ n           ┆ enue        ┆ ---   │
│ i64 ┆ i128     ┆ ---         ┆ ---         ┆   ┆ ---         ┆ ---         ┆ ---         ┆ str   │
│     ┆          ┆ str         ┆ str         ┆   ┆ str         ┆ str         ┆ str         ┆       │
╞═════╪══════════╪═════════════╪═════════════╪═══╪═════════════╪═════════════╪═════════════╪═══════╡
│ 0   ┆ 6192     ┆ 2023-09-01  ┆ 2023-09-01  ┆ … ┆ LIT         ┆ NEW         ┆ XASE        ┆ XASE  │
│     ┆          ┆ 09:33:02.87 ┆ 13:33:02.87 ┆   ┆             ┆             ┆             ┆       │
│     ┆          ┆ 3930097     ┆ 3930097     ┆   ┆             ┆             ┆             ┆       │
│ 1   ┆ 34803    ┆ 2023-09-01  ┆ 2023-09-01  ┆ … ┆ LIT         ┆ NEW        